In [ ]:
%run ../utils/logger

In [ ]:
%run ../utils/dataquality

## Init

In [ ]:
log_event("silver", "crm_products", "START", "Reading from bronze")

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

In [ ]:
df = spark.table("workspace.bronze.crm_prd_info")

## Transform data

In [0]:
# trim
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
# parse product key
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))
df.display()

In [0]:
# cost cleanup

df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

In [0]:
# normalisation

df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

In [0]:
#data casting
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

In [ ]:
# deduplicate: keep the most recent record per product
window = Window.partitionBy("prd_id").orderBy(col("prd_start_dt").desc())
df = df.withColumn("row_num", F.row_number().over(window))
df = df.filter(col("row_num") == 1).drop("row_num")

In [0]:
# rename friendly
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
     

In [0]:

df.limit(10).display()

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.crm_products")
row_count = check_not_empty(df, "silver.crm_products")
log_event("silver", "crm_products", "SUCCESS", "Write complete", rows=row_count)

In [ ]:
check_null_rate(df, "silver.crm_products", "product_id")
check_duplicates(df, ["product_id"])